In [116]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [117]:
df = pd.read_csv('/kaggle/train.txt',sep=';',header = None,names = ['text','emotions'])

In [118]:
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [119]:
df.isnull()

,text,emotions
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False
...,...,...
15995,False,False
15996,False,False
15997,False,False
15998,False,False


In [120]:
df.isnull().sum()

,0
text,0
emotions,0


In [121]:
unique_emotions = df['emotions'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i+=1

df['emotions'] = df['emotions'].map(emotion_numbers)



In [122]:
df

,text,emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [124]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [125]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [126]:
df['text'] = df['text'].apply(remove_punc)

In [127]:
def remove_numbers(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new + i
  return new
df['text'] = df['text'].apply(remove_numbers)

In [128]:
def remove_emojis(txt):
  new = ""
  for i in txt:
    if i.isascii():
      new += i
  return new

df['text'] = df['text'].apply(remove_emojis)


In [129]:
import nltk



In [130]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [131]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [132]:
stop_words = set(stopwords.words('english'))



In [133]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [140]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)




In [141]:
df['text'] = df['text'].apply(remove)

In [142]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [143]:
df.head()

,text,emotions
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [145]:
from sklearn.model_selection import train_test_split

In [146]:
x_train,x_test,y_train,y_test = train_test_split(df['text'],df['emotions'],test_size=0.2,random_state=42)

In [147]:
x_train

,text
676,refers course though cant help feeling somehow...
12113,im starting feel im suffering fatigue
7077,feel like probably would liked book little bit...
13005,really feel awkward
12123,im feeling little grumpy today lame weather te...
...,...
13418,love leave reader feeling confused slightly de...
5390,feel delicate
860,starting feel little stressed
15795,feel stressed tired worn shape neglected


In [148]:
x_test

,text
8756,ive made week feel beaten
4660,feel strategy worthwhile
6095,feel worthless weak say want find
304,feel clever nov
8241,im moved ive feeling kind gloomy
...,...
15578,feel useful pulpit find ironic often question ...
5746,dried bladders ready day im feeling brave
6395,feel thrilled matter days
7624,woke morning text mr c declaring walking work ...


In [151]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [152]:
bow_vectorizer = CountVectorizer()


In [153]:
x_train_bow = bow_vectorizer.fit_transform(x_train)
x_test_bow = bow_vectorizer.transform(x_test)

In [155]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [156]:
nb_model = MultinomialNB()
nb_model.fit(x_train_bow,y_train)

MultinomialNB()

In [160]:
pred_bow = nb_model.predict(x_test_bow)

In [161]:
print(accuracy_score(y_test,pred_bow))


0.768125


In [162]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [163]:
y_test

,emotions
8756,0
4660,5
6095,0
304,5
8241,0
...,...
15578,5
5746,5
6395,5
7624,5


In [168]:
tfidf_vectorizer = TfidfVectorizer()
x_train_tfidf = tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf = tfidf_vectorizer.transform(x_test)

nb2_model = MultinomialNB()
nb2_model.fit(x_train_tfidf,y_train)


MultinomialNB()

In [169]:
y_pred = nb2_model.predict(x_test_tfidf)

In [173]:
y_pred

array([0, 5, 0, ..., 5, 5, 0])

In [174]:
print(accuracy_score(y_test,y_pred))

0.6609375


In [178]:
from sklearn.linear_model import LogisticRegression

In [181]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(x_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [182]:
log_pred = logistic_model.predict(x_test_tfidf)

In [183]:
print(accuracy_score(y_test,log_pred))

0.8628125
